# JSON Handling

### 🧾 Python ↔ JSON Data Type Conversion Table

| Python Type               | JSON Type        | Notes                                                               |
|---------------------------|------------------|---------------------------------------------------------------------|
| `dict`                    | Object           | Key-value pairs                                                     |
| `list`, `tuple`           | Array            | Tuples are converted to lists; JSON does not have a tuple type      |
| `str`                     | String           | Direct mapping                                                      |
| `int`, `float`, `long`    | Number           | JSON numbers support both integers and decimals                     |
| `True` / `False`          | `true` / `false` | Boolean values                                                      |
| `None`                    | `null`           | Python `None` maps to JSON `null`                                   |

## Dictionary to JSON (Serialization/Encoding) String

In [10]:
import json

person = {"name":"Ajay", "age":33, "city":"Hyderabad", "hasChildren":True, "titles":["engineer","programmer","father"], \
          "children":({"firstName":"Saanvi", "age":7},{"firstName":"Anant", "age":3})}

personJSON = json.dumps(person, indent=2, sort_keys=True)
print(personJSON)

#personJSON = json.dumps(person, indent=4, separators=('; ', '= '))
#print(personJSON)

{
  "age": 33,
  "children": [
    {
      "age": 7,
      "firstName": "Saanvi"
    },
    {
      "age": 3,
      "firstName": "Anant"
    }
  ],
  "city": "Hyderabad",
  "hasChildren": true,
  "name": "Ajay",
  "titles": [
    "engineer",
    "programmer",
    "father"
  ]
}


## Dump Python Dictionary to File as JSON data

In [11]:
with open('person.json','w') as file:
    json.dump(person, file, indent=4)

In [14]:
! ls -l person.json

-rw-r--r--@ 1 ajay.dwivedi  staff  346 May 25 18:50 person.json


In [15]:
! cat person.json

{
    "name": "Ajay",
    "age": 33,
    "city": "Hyderabad",
    "hasChildren": true,
    "titles": [
        "engineer",
        "programmer",
        "father"
    ],
    "children": [
        {
            "firstName": "Saanvi",
            "age": 7
        },
        {
            "firstName": "Anant",
            "age": 3
        }
    ]
}

## Convert to Dictionary from JSON

In [16]:
person_converted = json.loads(personJSON)
print(person_converted)

{'age': 33, 'children': [{'age': 7, 'firstName': 'Saanvi'}, {'age': 3, 'firstName': 'Anant'}], 'city': 'Hyderabad', 'hasChildren': True, 'name': 'Ajay', 'titles': ['engineer', 'programmer', 'father']}


In [17]:
with open('person.json','r') as file:
    person_converted = json.load(file)
    print(person_converted)

{'name': 'Ajay', 'age': 33, 'city': 'Hyderabad', 'hasChildren': True, 'titles': ['engineer', 'programmer', 'father'], 'children': [{'firstName': 'Saanvi', 'age': 7}, {'firstName': 'Anant', 'age': 3}]}


## Define User Class

In [44]:
import json

class User:
    
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def __repr__(self):
        return f"User(name={self.name!r}, age={self.age})"
        

In [45]:
#!pip install colorama
#!conda install -c anaconda colorama

In [46]:
user = User('Ajay', 33)

In [3]:
# This is expected to throw error
try:
    userJSON = json.dumps(user)
except Exception as e:
    print(f'ERROR: {e}')

ERROR: Object of type User is not JSON serializable


## Method 01: Create Function based JSON Encoder/Decoder for Class Objects

In [6]:
def user_json_encode(o):
    if isinstance(o, User):
        return {'name': o.name, 'age': o.age, o.__class__.__name__: True}
    raise TypeError(f'Object of type {type(o)} is not JSON serializable')

def user_json_decode(dct):
    if User.__name__ in dct:
        return User(name=dct['name'], age=dct['age'])
    raise TypeError(f'Json data not of type User object')

In [25]:
# Encode user class object to JSON
userJSON = json.dumps(user, default=user_json_encode, indent=2)
print(userJSON)
print(type(userJSON))

{
  "name": "Ajay",
  "age": 33,
  "User": true
}
<class 'str'>


In [14]:
# By default, json.loads() converts the JSON string to Dict. NOT User object.
userDict = json.loads(userJSON)

print(userDict)
print(type(userDict))
#print(f'user.name: {userDict['name']}, user.age: {userDict['age']}')

{'name': 'Ajay', 'age': 33, 'User': True}
<class 'dict'>


In [28]:
# Decode user class object from JSON using Decoder function
userObj = json.loads(userJSON, object_hook=user_json_decode)
print(userObj)
type(userObj)

print(userObj.name)
print(userObj.age)

Ajay
33


## Method 02: Create JSON Encoder/Decoder for Class Objects Using JSONEncoder Module

In [47]:
import json
from json import JSONEncoder, JSONDecoder

class UserEncoder(JSONEncoder):
    def default(self, o):
        if isinstance(o, User):
            return {'name': o.name, 'age': o.age, o.__class__.__name__: True}
        return JSONEncoder.default(self, o)

class UserDecoder(JSONDecoder):
    def __init__(self, *args, **kwargs):
        super().__init__(object_hook=self.object_hook, *args, **kwargs)

    def object_hook(self, dct):
        if User.__name__ in dct:
            return User(name=dct['name'], age=dct['age'])

        return dct
    

In [38]:
# Calling method 01
userJSON = UserEncoder().encode(user)
print(userJSON)
print(type(userJSON))
print(json.loads(userJSON))

print ('*'*20)

# Calling method 02
userJSON = json.dumps(user, cls=UserEncoder)
print(userJSON)
print(type(userJSON))
print(json.loads(userJSON))

{"name": "Ajay", "age": 33, "User": true}
<class 'str'>
{'name': 'Ajay', 'age': 33, 'User': True}
********************
{"name": "Ajay", "age": 33, "User": true}
<class 'str'>
{'name': 'Ajay', 'age': 33, 'User': True}


In [43]:
# decode using previously created decoder function
userObj = json.loads(userJSON, object_hook=user_json_decode)
print(userObj)
type(userObj)

print(userObj.name)
print(userObj.age)

Ajay
33


In [50]:
# Decode user class object from JSON using Decoder function
userObj = json.loads(userJSON, cls=UserDecoder)

print(userObj)
print(type(userObj))
print(userObj.name)
print(userObj.age)

User(name='Ajay', age=33)
<class '__main__.User'>
Ajay
33
